In [ ]:
import torch
import pandas as pd
import sklearn
import altair as alt
import numpy as np
import yaml
from omegaconf import OmegaConf
from ts2.train.main_cell_inference import SingleCellTempInferenceDataset
from tqdm import tqdm
import gzip
import pydicom
from PIL import Image
import gc
import matplotlib.pyplot as plt

In [ ]:
from torchsrh.utils.open_color import OpenColor

In [ ]:
from collections import defaultdict

def merge_list_of_dicts(dict_list):
    merged = defaultdict(list)
    for d in dict_list:
        for k, v in d.items():
            merged[k].extend(v)
    return dict(merged)

In [ ]:
from ts2.data.db_improc import MemmapTileReader
from ts2.data.transforms import HistologyTransform
from ts2.train.main_cell_inference import SingleCellListInferenceDataset
import io
import base64
import einops
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
from PIL import Image
def im_to_bytestr(image) -> str:
        output = io.BytesIO()
        Image.fromarray(image).save(output, format='JPEG')
        return "data:image/jpeg;base64," + base64.b64encode(
            output.getvalue()).decode()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
# ours:
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/941f49cb_May29-01-43-17_sd1000_smpt_BENCHDB_SE_epoch0-step124999_tune0/predictions/train_predictions.pt.gz") as fd:

# dv2
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/0e97d769_May26-11-14-43_sd1000_smpt_BENCHDB_SE_epoch0-step124999_tune3/predictions/train_predictions.pt.gz") as fd:
#    db_data = torch.load(fd)

#db_data = pd.DataFrame({
#    "path":db_data["path"],
#    "label":db_data["label"],
#    "embeddings": [i for i in db_data["embeddings"]]
#})
#db_embs = torch.stack(db_data["embeddings"].tolist())


#db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/2510a2cb_May30-19-10-14_sd1000_INFDB_dev/predictions/pred.pt")
#db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/17b1bb09_May30-19-44-14_sd1000_INFDB_dev/predictions/pred.pt")

# g2m, ours, with instance norm
##db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/e2e43218_Jun03-18-51-29_sd1000_INFDB_dev/predictions/pred.pt")

# g2m, ours, no instance norm
db_data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/dd7f97e0_Jun06-21-19-30_sd1000_INFDB_NOIN_dev/predictions/pred.pt")

db_data = pd.DataFrame(db_data)
db_embs = torch.stack(db_data["embeddings"].tolist())

In [ ]:
ttype_ = pd.read_csv("data/srh_ttype_cheng.csv")

In [ ]:
def sample_idx(group, n=8192):
    return group.sample(n=min(n, len(group)), random_state=1000).index

sampled_idx = db_data.groupby("label").apply(sample_idx).explode().values

In [ ]:
cell_samples = sorted(sampled_idx)

In [ ]:
db_sample = db_data.iloc[cell_samples]
db_sample_embs = torch.stack(db_sample["embeddings"].tolist()).numpy()

In [ ]:
all_embs = torch.nn.functional.normalize(torch.tensor(db_sample_embs))

In [ ]:
tsne = sklearn.manifold.TSNE(n_components=2, perplexity=50)
embeddings_2d = tsne.fit_transform(all_embs)


In [ ]:
min_vals = embeddings_2d.min(axis=0)
max_vals = embeddings_2d.max(axis=0)
embeddings_2d = (embeddings_2d - min_vals) / (max_vals - min_vals) * 0.9 + 0.05

In [ ]:

with open("../train/config/chengjia/inference_dinov2_scsrhdb.yaml") as fd:
    cf = OmegaConf.create(yaml.safe_load(fd))
cf.data.test_dataset.params.cell_instances = "../train/srhum_glioma_2m_.csv"

dataset = SingleCellListInferenceDataset(
     transform=HistologyTransform(**cf.data.xform_params),
    **cf.data.test_dataset.params)

In [ ]:
images = [dataset[i]["image"] for i in cell_samples]

In [ ]:
im_str = [im_to_bytestr(einops.rearrange(
    (adjust_contrast(adjust_brightness(i.squeeze(),2),2) * 255).to(torch.uint8), "c h w -> h w c").numpy() )for i in images]

In [ ]:
combined_data = pd.DataFrame({
    "x": embeddings_2d[:,0],
    "y": embeddings_2d[:,1],
    "path":db_data.iloc[cell_samples]["path"],
    "image": im_str,
    "label": db_data.iloc[cell_samples]["label"],
})


In [ ]:
combined_data["nio_num"] = combined_data["path"].str.extract("(NIO_UM_[0-9]+)")


In [ ]:
combined_data = combined_data.merge(
    ttype_,
    left_on="nio_num", right_on="type_institution_number", how="left"
).drop("type_institution_number", axis=1
).rename({"ttype_cheng": "weak_label"}, axis=1)
#combined_data.loc[combined_data["label"]=="normal", "slide_label"] = "normal"


In [ ]:
combined_data.loc[combined_data["label"]=="normal", "weak_label"] = "normal"
combined_data["weak_label"] = combined_data["weak_label"].fillna("other")

In [ ]:
weak_labels = combined_data["weak_label"].drop_duplicates().tolist()
#colors = ['#65a30d', '#eab308',  '#0891b2', '#dc2626', '#ea580c', '#475569', '#db2777', '#7c3aed', '#c026d3']
colors = ['#65a30d', "#dc2626"]

In [ ]:
# initial tSNE with tumor type

alt.data_transformers.disable_max_rows()
tsne_unit_axis = alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6),
                          domain=False,
                          labels=False,
                          title="")

base_chart = alt.Chart(combined_data).mark_point(
    filled=True
).encode(
    x=alt.X("x",
            axis=tsne_unit_axis,
            scale=alt.Scale(domain=[0,1])),
    y=alt.Y("y",
            axis=tsne_unit_axis,
            scale=alt.Scale(domain=[0,1])),
    tooltip=["image", "path"],
    color=alt.Color("weak_label:N",
                      scale=alt.Scale(domain=weak_labels,
                                      range=colors)),
    opacity=alt.value(.8)
)

In [ ]:
base_chart.properties(height=600,width=600).interactive().save("silica_eval/out/g2m_ours_noln.png")

# GMM fitting

In [ ]:
db_mean = db_embs.mean(dim=0)
db_embs = db_embs - db_mean
db_embs_norm = torch.nn.functional.normalize(db_embs, dim=1)

bic_scores = []
aic_scores = []
k_range = [2,8,16,24,32,64,128,256]
gmms = []
for k in k_range:
    print(f"Fitting GMM with K={k}")
    gmm = GaussianMixture(
        n_components=k,
        covariance_type='diag',
        max_iter=200,
        init_params='kmeans',
        random_state=0)
    
    gmm.fit(db_embs_norm)
    gmms.append(gmm)
    
    bic = gmm.bic(db_embs_norm)
    aic = gmm.aic(db_embs_norm)

    bic_scores.append(bic)
    aic_scores.append(aic)
    
    print(bic)
    print(aic)


In [ ]:
#sampled_embs = X_scaled[cell_samples]

In [ ]:
#labels = gmms[-1].predict(sampled_pca)

In [ ]:
cluster_positive_rates = []
cluster_size = []
for k, g in zip(k_range, gmms):
    gmm_pred = g.predict(db_embs_norm)
    gmm_pred_df = pd.DataFrame({"cluster": gmm_pred, "label": db_data["label"]=="tumor"})
    cluster_positive_rates.append(gmm_pred_df.groupby("cluster").mean()["label"].tolist())
    cluster_size.append(gmm_pred_df.groupby("cluster").count()["label"].tolist())
    
    combined_data2 = pd.DataFrame({
        "x": embeddings_2d[:,0],
        "y": embeddings_2d[:,1],
        "path":db_data.iloc[cell_samples]["path"],
        "image": im_str,
        "gt": db_data.iloc[cell_samples]["label"],
        "cluster": gmm_pred[cell_samples]
    })

    legend_selection = alt.selection_point(fields=["label"], bind="legend")

    alt.data_transformers.disable_max_rows()
    tsne_unit_axis = alt.Axis(tickSize=0,
                              values=np.linspace(0,1,6),
                              domain=False,
                              labels=False,
                              title="")

    base_chart = alt.Chart(combined_data2).mark_point(
        filled=True
    ).encode(
        x=alt.X("x",
                axis=tsne_unit_axis,
                scale=alt.Scale(domain=[0,1])),
        y=alt.Y("y",
                axis=tsne_unit_axis,
                scale=alt.Scale(domain=[0,1])),
        tooltip=["image", "path", "cluster", "gt"],
        color=alt.Color("cluster:N",
                         ),
        opacity=alt.condition(legend_selection, alt.value(0.7), alt.value(0.05))
    ).add_params(legend_selection)

    base_chart.properties(width=600,height=600).interactive().save(f"silica_gmm/tsne/gmm_g2m_m{k}_tsne.png")
    base_chart.properties(width=600,height=600).interactive().save(f"silica_gmm/tsne/gmm_g2m_m{k}_tsne.pdf")
    base_chart.properties(width=600,height=600).interactive().save(f"silica_gmm/tsne/gmm_g2m_m{k}_tsne.html")
    base_chart.properties(width=600,height=600).interactive().save(f"silica_gmm/tsne/gmm_g2m_m{k}_tsne.json")

In [ ]:
import joblib
for k, g in zip(k_range, gmms):
    joblib.dump(g, f"silica_gmm/models/gmm_g2m_m{k}.pkl")

In [ ]:
metrics = pd.DataFrame({"k":k_range, "AIC":aic_scores, "BIC":bic_scores, "pos_rate": cluster_positive_rates, "cluster_size": cluster_size})
metrics.to_csv("silica_gmm/stats/gmm_g2m_metrics.csv", index=False)
metrics = pd.melt(metrics, id_vars=['k'], value_vars=['AIC', "BIC"], var_name='metric', value_name='val')

In [ ]:
xaxis = alt.Axis(tickSize=0,
                          values=[0,64,128,192,256],
                          title="Number of mixtures")
yaxis = alt.Axis(tickSize=0,
                          title="Metric (x 1.0e9)")
alt.Chart(metrics).mark_line(point=True).transform_calculate(
    tval='datum.val / 1000000000'
).encode(
    x=alt.X("k", axis=xaxis, scale=alt.Scale(domain=[0,256])),
    y=alt.Y("tval:Q",axis=yaxis, scale=alt.Scale(zero=False)),
    color="metric:N"
).interactive().save("silica_gmm/stats/gmm_g2m_metrics.png")

In [ ]:
mean_conf = [np.abs([c-0.5 for c in p]).mean()*2 for p in cluster_positive_rates]
alt.Chart(pd.DataFrame({"k":k_range, "mean_conf": mean_conf})).mark_line(point=True).encode(
    x=alt.X("k", axis=xaxis, scale=alt.Scale(domain=[0,256])),
    y=alt.Y("mean_conf:Q",axis= alt.Axis(tickSize=0,
                          title="Mean mixture confidence"), scale=alt.Scale(zero=False)),
).interactive().interactive().save("silica_gmm/stats/gmm_g2m_cluster_confidence.pdf")

In [ ]:
mean_conf

In [ ]:
def topk_random_tiebreak(g):
    g = g.sample(frac=1, random_state=None)  # shuffle group
    ranked = g.assign(rank=g['prob'].rank(method='first', ascending=False))
    return ranked[ranked['rank'] <= 5]


def pad_with_transparency(images_rgb: np.ndarray, pad: int = 1) -> np.ndarray:
    n, h, w, _ = images_rgb.shape
    padded = np.zeros((n, h + 2 * pad, w + 2 * pad, 4), dtype=images_rgb.dtype)

    # Insert RGB into center
    padded[:, pad:-pad, pad:-pad, :3] = images_rgb
    padded[:, pad:-pad, pad:-pad, 3] = 255  # Alpha channel: opaque in center

    return padded



In [ ]:
for k, g in zip(k_range, gmms):

    gmm_pred = g.predict_proba(db_embs_norm)

    combined_data2 = pd.DataFrame({
        "path":db_data["path"],
        "gt": db_data["label"],
        "cluster": gmm_pred.argmax(axis=1),
        "prob": gmm_pred.max(axis=1)
    })

    center5 = combined_data2.groupby("cluster", group_keys=False).apply(topk_random_tiebreak)
    rand5 = combined_data2.groupby("cluster", group_keys=False).sample(5)
    rand5["rank"]=100

    allsample = pd.concat([center5, rand5])
    allsample = allsample.sort_values(["cluster", "rank"])
    images = [dataset[i]["image"] for i in allsample.index.tolist()]
    im_str = [im_to_bytestr(einops.rearrange(
        (adjust_contrast(adjust_brightness(i.squeeze(),2),2) * 255).to(torch.uint8), "c h w -> h w c").numpy() )for i in images]

    im_array = np.stack([einops.rearrange(
        (adjust_contrast(adjust_brightness(i.squeeze(),2),2) * 255).to(torch.uint8), "c h w -> h w c").numpy() for i in images])

    im_display = einops.rearrange(pad_with_transparency(im_array, pad=2), "(b n) h w c -> (n h) (b w) c", b=k)
    Image.fromarray(im_display).save(f"silica_gmm/mixture_samples/gmm{k}_clusters.png")


In [ ]:
    alt.data_transformers.disable_max_rows()
    tsne_unit_axis = alt.Axis(tickSize=0,
                              values=np.linspace(0,1,6),
                              domain=False,
                              labels=False,
                              title="")

    base_chart = alt.Chart(combined_data2).mark_point(
        filled=True
    ).encode(
        x=alt.X("x",
                axis=tsne_unit_axis,
                scale=alt.Scale(domain=[0,1])),
        y=alt.Y("y",
                axis=tsne_unit_axis,
                scale=alt.Scale(domain=[0,1])),
        tooltip=["image", "path", "cluster", "gt"],
        color=alt.Color("cluster:N",
                         ),
        opacity=alt.condition(legend_selection, alt.value(0.7), alt.value(0.05))
    ).add_params(legend_selection)

In [ ]:
import matplotlib
import numpy as np
def get_mpl_colormap_hex_list(cmap_name='RdYlGn', n=256):
    """
    Sample `n` colors from a matplotlib colormap and return as hex codes.

    Args:
        cmap_name (str): Name of the matplotlib colormap.
        n (int): Number of colors to sample.

    Returns:
        List[str]: List of hex color codes.
    """
    cmap = matplotlib.cm.get_cmap(cmap_name, n)
    return [matplotlib.colors.rgb2hex(cmap(i)) for i in range(cmap.N)]

# Example usage
hex_colors = get_mpl_colormap_hex_list('RdYlGn', 256)

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

In [ ]:
data = pd.DataFrame({"pos_rate": [0.12320829961854121, 0.9424717444092706, 0.007920075582447183, 0.8321623139141388, 0.7764409011117612, 0.8806957733054731, 0.3710393490774659, 0.01834528807060457, 0.5990491782633336, 0.8092892033013306, 0.1311596266943686, 0.06322619586539913, 0.05968638539892677, 0.008541718247090864, 0.9978847170809095, 0.01797972450788744, 0.2271186440677966, 0.13576392563586975, 0.962493911349245, 0.1978056204431231, 0.9208270920518246, 0.9828266877220687, 0.942565221704196, 0.00645079914229598, 0.7187105702307787, 0.24500006910754515, 0.848616309402291, 0.004890418402463322, 0.9538893290185705, 0.7204800181138911, 0.10304759541826904, 0.5878180623581412],
"incidence": [69208, 78657, 49747, 65897, 54688, 54558, 59239, 60506, 63524, 71244, 59683, 53016, 62242, 48468, 56730, 43994, 68735, 56694, 90332, 70635, 64757, 65858, 60486, 55497, 73707, 72351, 70536, 49689, 74473, 35332, 72549, 66968]})

data["incidence"] /= 1000


In [ ]:
data["pos_rate"].iloc[4] = 0
data["pos_rate"].iloc[-3] = 0

In [ ]:
data["cluster"] = np.arange(32)

In [ ]:
base_chart = alt.Chart(data).mark_bar(width=16)

pos_rate = base_chart.encode(
    x=alt.X("cluster",
            scale=alt.Scale(domain=[-0.5,31.5]),
            axis=alt.Axis(tickSize=0,
                              values=np.linspace(0,31,32),
                              domain=False,
                              labels=False,
                              title="")),
    y=alt.Y("pos_rate", axis=alt.Axis(tickSize=0,
                              #values=np.linspace(0,31,32),
                              domain=False,
                              labels=True,
                              title="Tumor rate")),
    color=alt.Color('pos_rate:Q', scale=alt.Scale(domain=[1, 0],
                                            range=hex_colors),  legend=None)
).properties(height=60, width=800)

incidence = base_chart.encode(
    x=alt.X("cluster",
            scale=alt.Scale(domain=[-0.5,31.5]),
            axis=alt.Axis(tickSize=0,
                              values=np.linspace(0,31,32),
                              domain=False,
                              labels=True,
                              title="Cluster")),
    y=alt.Y("incidence", axis=alt.Axis(tickSize=0,
                              #values=np.linspace(0,31,32),
                              domain=False,
                              labels=True,
                              title="Cells (K)")),
    color=alt.value("#475569")
).properties(height=60, width=800)

(pos_rate & incidence).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_legend(titleFontSize=14).interactive().save(f"silica_gmm/cluster32_stats_manual.pdf")